In [1]:
import heapq
import math

# Hücre koordinatları (row, col)
def heuristic(a, b, kind="manhattan"):
    (x1, y1), (x2, y2) = a, b
    if kind == "euclidean":
        return math.hypot(x2 - x1, y2 - y1)
    # default: manhattan
    return abs(x1 - x2) + abs(y1 - y2)

def neighbors(node, grid, allow_diagonal=False):
    (r, c) = node
    rows, cols = len(grid), len(grid[0])
    steps = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    if allow_diagonal:
        steps += [(-1, -1), (-1, 1), (1, -1), (1, 1)]
    result = []
    for dr, dc in steps:
        nr, nc = r + dr, c + dc
        if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 0:
            result.append((nr, nc))
    return result

def reconstruct_path(came_from, current):
    path = [current]
    while current in came_from:
        current = came_from[current]
        path.append(current)
    path.reverse()
    return path

def astar(grid, start, goal, heuristic_kind="manhattan", allow_diagonal=False):
    """
    grid: 2D list where 0 = free cell, 1 = obstacle
    start, goal: (row, col)
    returns: path (list of nodes) or None if no path
    """
    open_set = []
    # heap elements: (f_score, g_score, node)
    g_score = {start: 0}
    f_score = {start: heuristic(start, goal, heuristic_kind)}
    heapq.heappush(open_set, (f_score[start], g_score[start], start))

    came_from = {}

    closed_set = set()

    while open_set:
        _, current_g, current = heapq.heappop(open_set)

        if current == goal:
            return reconstruct_path(came_from, current)

        if current in closed_set:
            continue
        closed_set.add(current)

        for neighbor in neighbors(current, grid, allow_diagonal):
            tentative_g = g_score[current] + math.hypot(neighbor[0]-current[0], neighbor[1]-current[1])
            # if only 4-directional, distance is 1; with diagonal we used hypot (√2)
            if neighbor in g_score and tentative_g >= g_score[neighbor]:
                continue  # not a better path

            # this path is the best until now
            came_from[neighbor] = current
            g_score[neighbor] = tentative_g
            f = tentative_g + heuristic(neighbor, goal, heuristic_kind)
            f_score[neighbor] = f
            heapq.heappush(open_set, (f, tentative_g, neighbor))

    return None  # no path found

def print_grid_with_path(grid, path, start, goal):
    chars = {0: "·", 1: "█"}
    grid_vis = [[chars[cell] for cell in row] for row in grid]
    if path:
        for (r, c) in path:
            if (r, c) == start:
                grid_vis[r][c] = "S"
            elif (r, c) == goal:
                grid_vis[r][c] = "G"
            else:
                grid_vis[r][c] = "*"
    for row in grid_vis:
        print(" ".join(row))

if __name__ == "__main__":
    # Örnek ızgara: 0 = boş, 1 = engel
    example_grid = [
        [0,0,0,0,0,0,0,0],
        [0,1,1,1,0,1,1,1],
        [0,0,0,1,0,1,0,0],
        [0,1,0,0,0,1,0,0],
        [0,1,0,1,0,0,0,0],
        [0,0,0,1,0,1,1,0],
        [0,1,0,0,0,0,0,1],
        [0,0,0,0,1,0,0,0],
    ]
    start = (0, 0)
    goal = (7, 7)

    path = astar(example_grid, start, goal, heuristic_kind="manhattan", allow_diagonal=False)
    print("Bulunan yol (satır, sütun) biçiminde:", path)
    print()
    print_grid_with_path(example_grid, path, start, goal)

Bulunan yol (satır, sütun) biçiminde: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (1, 4), (2, 4), (3, 4), (4, 4), (5, 4), (6, 4), (6, 5), (6, 6), (7, 6), (7, 7)]

S * * * * · · ·
· █ █ █ * █ █ █
· · · █ * █ · ·
· █ · · * █ · ·
· █ · █ * · · ·
· · · █ * █ █ ·
· █ · · * * * █
· · · · █ · * G


In [2]:
import pygame
import math

# Ayarlar
SCREEN_WIDTH = 1200
SCREEN_HEIGHT = 800
CAR_WIDTH = 40
CAR_HEIGHT = 80

class OtonomArac:
    def __init__(self, x, y):
        self.x = x
        self.y = y
        self.angle = 0  # Aracın yönü (derece)
        self.velocity = 0
        self.steering_angle = 0 # Direksiyon açısı
        
        # Orijinal resim (Sprite)
        self.original_image = pygame.Surface((CAR_WIDTH, CAR_HEIGHT), pygame.SRCALPHA)
        pygame.draw.rect(self.original_image, (0, 0, 255), (0, 0, CAR_WIDTH, CAR_HEIGHT))
        self.image = self.original_image

    def update(self):
        # Basit Kinematik Model (Bicycle Model)
        # Hız ve yöne göre yeni konumu hesapla
        self.x += self.velocity * math.sin(math.radians(self.angle))
        self.y -= self.velocity * math.cos(math.radians(self.angle))
        
        # Direksiyon açısına göre aracın dönmesi
        # Gerçekte: açı += (hız / tekerlek_mesafesi) * tan(direksiyon_açısı)
        self.angle += self.steering_angle * self.velocity * 0.1 

    def draw(self, screen):
        # Aracı döndürerek çiz
        rotated_image = pygame.transform.rotate(self.original_image, -self.angle)
        rect = rotated_image.get_rect(center=(self.x, self.y))
        screen.blit(rotated_image, rect.topleft)
        
        # Sensörleri çiz (Lidar Simülasyonu)
        self.draw_sensors(screen)

    def draw_sensors(self, screen):
        # 5 adet Raycast (Işın) simülasyonu
        sensor_angles = [-30, -15, 0, 15, 30]
        sensor_length = 150
        
        for s_angle in sensor_angles:
            rad_angle = math.radians(self.angle + s_angle)
            end_x = self.x + math.sin(rad_angle) * sensor_length
            end_y = self.y - math.cos(rad_angle) * sensor_length
            pygame.draw.line(screen, (0, 255, 0), (self.x, self.y), (end_x, end_y), 1)

# Oyun Döngüsü
pygame.init()
screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
clock = pygame.time.Clock()
arac = OtonomArac(SCREEN_WIDTH/2, SCREEN_HEIGHT/2)

running = True
while running:
    screen.fill((50, 50, 50)) # Asfalt rengi
    
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # Manuel Kontrol (Test için)
    keys = pygame.key.get_pressed()
    if keys[pygame.K_UP]: arac.velocity += 0.1
    elif keys[pygame.K_DOWN]: arac.velocity -= 0.1
    else: arac.velocity *= 0.95 # Sürtünme
    
    if keys[pygame.K_LEFT]: arac.steering_angle = -5
    elif keys[pygame.K_RIGHT]: arac.steering_angle = 5
    else: arac.steering_angle = 0

    arac.update()
    arac.draw(screen)
    
    pygame.display.flip()
    clock.tick(60)

pygame.quit()

C:\Users\90534\AppData\Roaming\Python\Python312\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.12.6)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [3]:
import pygame
import sys

# --- Ayarlar ---
EKRAN_GENISLIK = 800
EKRAN_YUKSEKLIK = 600
FPS = 60
ARKA_PLAN_RENGI = (30, 30, 30) # Koyu gri
DIREKSIYON_RENGI = (200, 200, 200)
ISARET_RENGI = (255, 0, 0) # Dönüşü görmek için kırmızı şerit

# --- Pygame Başlatma ---
pygame.init()
screen = pygame.display.set_mode((EKRAN_GENISLIK, EKRAN_YUKSEKLIK))
pygame.display.set_caption("2D Direksiyon Simülasyonu")
clock = pygame.time.Clock()

def direksiyon_olustur(cap):
    """
    Basit bir direksiyon yüzeyi (Surface) oluşturur.
    Resim yüklemek yerine çizim yaparak gösteriyoruz.
    """
    # Şeffaf bir yüzey oluştur
    surface = pygame.Surface((cap, cap), pygame.SRCALPHA)
    
    # 1. Dış Çember (Simit)
    pygame.draw.circle(surface, DIREKSIYON_RENGI, (cap//2, cap//2), cap//2, 20)
    
    # 2. İç Göbek
    pygame.draw.circle(surface, DIREKSIYON_RENGI, (cap//2, cap//2), 30)
    
    # 3. Kollar (Spokes) - Sol, Sağ ve Alt
    pygame.draw.rect(surface, DIREKSIYON_RENGI, (0, cap//2 - 10, cap, 20)) # Yatay kol
    pygame.draw.rect(surface, DIREKSIYON_RENGI, (cap//2 - 10, cap//2, 20, cap//2)) # Dikey alt kol

    # 4. Üstteki Kırmızı İşaret (Dönüşü anlamak için)
    pygame.draw.rect(surface, ISARET_RENGI, (cap//2 - 10, 0, 20, 30))
    
    return surface

# --- Hazırlık ---
# Direksiyonu bir kez oluşturuyoruz (veya pygame.image.load('resim.png') ile yükleyebilirsiniz)
orijinal_direksiyon = direksiyon_olustur(300)
direksiyon_rect = orijinal_direksiyon.get_rect(center=(EKRAN_GENISLIK//2, EKRAN_YUKSEKLIK//2))

aci = 0          # Mevcut açı
donus_hizi = 4   # Tuşa basınca ne kadar hızlı döneceği
toplanma_hizi = 2 # Tuşu bırakınca merkeze dönme hızı (Force Feedback simülasyonu gibi)
max_aci = 540    # Maksimum dönüş açısı (örn. 1.5 tur)

running = True
while running:
    # --- 1. Olayları Dinle ---
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # --- 2. Tuş Kontrolleri ---
    keys = pygame.key.get_pressed()
    
    # Sola Dönüş (Açıyı artır)
    if keys[pygame.K_LEFT] or keys[pygame.K_a]:
        if aci < max_aci:
            aci += donus_hizi
            
    # Sağa Dönüş (Açıyı azalt)
    elif keys[pygame.K_RIGHT] or keys[pygame.K_d]:
        if aci > -max_aci:
            aci -= donus_hizi
            
    # Tuşa basılmıyorsa direksiyonu yavaşça merkeze topla (Opsiyonel)
    else:
        if aci > 0:
            aci -= toplanma_hizi
            if aci < 0: aci = 0
        elif aci < 0:
            aci += toplanma_hizi
            if aci > 0: aci = 0

    # --- 3. Hesaplama ve Döndürme ---
    # Not: Pygame'de pozitif açı saatin tersi yönüdür (Counter-Clockwise)
    
    # Orijinal resmi döndürerek yeni bir yüzey oluşturuyoruz
    donmus_direksiyon = pygame.transform.rotate(orijinal_direksiyon, aci)
    
    # Yeni yüzeyin merkezini, eski yüzeyin merkezine eşitliyoruz.
    # BU ADIM ÇOK KRİTİKTİR. Yapmazsanız direksiyon ekranın sol üstüne doğru kayar.
    yeni_rect = donmus_direksiyon.get_rect(center=direksiyon_rect.center)

    # --- 4. Çizim ---
    screen.fill(ARKA_PLAN_RENGI)
    
    # Döndürülmüş resmi yeni koordinatına çiz
    screen.blit(donmus_direksiyon, yeni_rect)
    
    # Bilgi Yazısı (Debug)
    font = pygame.font.SysFont("Arial", 18)
    yazi = font.render(f"Açı: {int(aci)} derece", True, (255, 255, 255))
    screen.blit(yazi, (10, 10))

    pygame.display.flip()
    clock.tick(FPS)

pygame.quit()
sys.exit()

SystemExit: 

C:\Users\90534\AppData\Roaming\Python\Python312\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
import pygame
import sys

# --- Ayarlar ---
EKRAN_GENISLIK = 800
EKRAN_YUKSEKLIK = 600
FPS = 60

# Renk Tanımları
RENK_CIM = (34, 139, 34)       # Arka plan
RENK_YOL = (100, 100, 100)     # Gri asfalt
RENK_PARK_ZEMIN = (120, 120, 120) # Park alanı biraz daha farklı gri
RENK_PARK_CIZGI = (220, 220, 220) # Park yeri çizgileri
RENK_ENGEL = (200, 50, 50)     # Kırmızı engeller
RENK_OYUNCU = (50, 100, 250)   # Mavi oyuncu karesi

# --- Pygame Başlatma ---
pygame.init()
screen = pygame.display.set_mode((EKRAN_GENISLIK, EKRAN_YUKSEKLIK))
pygame.display.set_caption("2D Sabit Harita: Yollar, Park ve Engeller")
clock = pygame.time.Clock()

# --- HARİTA TANIMLARI (Dikdörtgen Listeleri) ---

# 1. Yollar (Asfalt alanlar)
# Rect(x, y, genişlik, yükseklik) formatında tanımlıyoruz.
yollar = [
    pygame.Rect(50, 50, 700, 80),   # Üst ana yol
    pygame.Rect(50, 470, 700, 80),  # Alt ana yol
    pygame.Rect(50, 130, 80, 340),  # Sol bağlantı yolu
    pygame.Rect(670, 130, 80, 340), # Sağ bağlantı yolu
    pygame.Rect(350, 250, 100, 220) # Ortadaki parka giden yol
]

# 2. Park Alanı
park_alani_ana = pygame.Rect(250, 150, 300, 150) # Parkın kendisi
park_yerleri_cizgileri = []
# Park alanının içine dikey çizgiler çizerek park yerlerini belirleyelim
for i in range(5):
    x_pos = 250 + (i + 1) * 50 # Her 50 pikselde bir çizgi
    # Çizgileri Rect olarak değil, çizim anında line olarak çizeceğiz
    park_yerleri_cizgileri.append(x_pos)


# 3. Engeller (İçinden geçilemeyecek alanlar)
engeller = [
    pygame.Rect(300, 70, 40, 40),   # Üst yolda bir kaza/kutu
    pygame.Rect(690, 300, 40, 80),  # Sağ yolda bir bariyer
    pygame.Rect(200, 200, 30, 30),  # Çimlerin üzerinde bir kaya
    pygame.Rect(360, 350, 80, 20)   # Park girişini daraltan bir engel
]

# --- OYUNCU TANIMI (Test için) ---
oyuncu = pygame.Rect(100, 80, 30, 30) # 30x30'luk bir kare
oyuncu_hizi = 4

def haritayi_ciz():
    """Tanımladığımız listeleri kullanarak haritayı ekrana çizer."""
    screen.fill(RENK_CIM) # En alta çim zemini döşe

    # Yolları çiz
    for yol in yollar:
        pygame.draw.rect(screen, RENK_YOL, yol)

    # Park alanını çiz
    pygame.draw.rect(screen, RENK_PARK_ZEMIN, park_alani_ana)
    # Park çizgilerini çiz (Sadece görsel)
    for x_pos in park_yerleri_cizgileri:
        # Park alanının üstünden altına çizgiler
        start_pos = (x_pos, park_alani_ana.top + 5)
        end_pos = (x_pos, park_alani_ana.bottom - 5)
        pygame.draw.line(screen, RENK_PARK_CIZGI, start_pos, end_pos, 3)

    # Engelleri çiz
    for engel in engeller:
        pygame.draw.rect(screen, RENK_ENGEL, engel)


# --- Ana Döngü ---
running = True
while running:
    # 1. Olay Kontrolü
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # 2. Oyuncu Hareketi ve Çarpışma Kontrolü
    keys = pygame.key.get_pressed()
    hareket_x = 0
    hareket_y = 0

    if keys[pygame.K_LEFT]:  hareket_x = -oyuncu_hizi
    if keys[pygame.K_RIGHT]: hareket_x = oyuncu_hizi
    if keys[pygame.K_UP]:    hareket_y = -oyuncu_hizi
    if keys[pygame.K_DOWN]:  hareket_y = oyuncu_hizi

    # --- Kritik Kısım: Çarpışma Öncesi Kontrol ---
    # Oyuncuyu hemen hareket ettirmiyoruz. Önce "hareket ederse nerede olacak"
    # diye sanal bir dikdörtgen oluşturuyoruz.
    gelecek_konum_x = oyuncu.move(hareket_x, 0)
    gelecek_konum_y = oyuncu.move(0, hareket_y)

    # X ekseninde çarpışma var mı?
    carpisma_x = False
    for engel in engeller:
        if gelecek_konum_x.colliderect(engel):
            carpisma_x = True
            break # Bir engele çarptıysak diğerlerine bakmaya gerek yok
    
    # Y ekseninde çarpışma var mı?
    carpisma_y = False
    for engel in engeller:
        if gelecek_konum_y.colliderect(engel):
            carpisma_y = True
            break

    # Eğer çarpışma YOKSA, oyuncunun gerçek konumunu güncelle
    if not carpisma_x:
        oyuncu.x += hareket_x
    if not carpisma_y:
        oyuncu.y += hareket_y

    # Ekran dışına çıkmayı engelle
    oyuncu.clamp_ip(screen.get_rect())


    # 3. Çizim İşlemleri
    haritayi_ciz() # Önce harita
    pygame.draw.rect(screen, RENK_OYUNCU, oyuncu) # Sonra oyuncu (haritanın üstünde)

    pygame.display.flip()
    clock.tick(FPS)

pygame.quit()
sys.exit()

SystemExit: 